## **Retrieval Extraction with Embeddings and Cosine Similarity (Without Generation)**

Retrieval: Retrieve the most relevant passage/context based using embeddings and cosine similarity.

Extraction: Extract the most relevant answer based on embeddings and cosine similarity, without any augmentation or generation steps.

Result: The answer is extracted directly from the context provided by the retrieved passage. The answer is the most relevant passage based on similarity.There is no generative model involved.

## Embeddings

**all-MiniLM-L6-v2**: A lightweight, transformer-based model for generating efficient and high-quality sentence embeddings, good for general-purpose NLP tasks.

In [ ]:
!pip install -qqq ollama
!pip install -qqq langchain faiss-cpu python-dotenv
!pip install -qqq -U langchain-openai
!pip install -qqq langchain-core langchain-openai langchain-community
!pip install -qqq matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.0 MB/s eta 0:00:00


In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
import requests
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline, AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import ollama
import torch
from langchain_community.embeddings import OllamaEmbeddings
import subprocess
import json
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import DistilBertForQuestionAnswering, DistilBertTokenizer
from transformers import BartTokenizer, BartForConditionalGeneration
# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Ensure directory exists
#os.makedirs('./data/', exist_ok=True)

In [ ]:
# Load the summary from the file
def load_summary(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

# Main execution
summary_file_path = './data/msc_ai_hullonline_short.txt'
loaded_summary = load_summary(summary_file_path)

# Print the loaded summary
print(loaded_summary)

The MSc Artificial Intelligence Online is delivered 100% online.
The MSc Artificial Intelligence Online program takes two years to complete on a part-time basis.
The total cost of the the MSc Artificial Intelligence Online program is £8,950, with instalment and funding options available.
The start dates for the for the MSc Artificial Intelligence Online program are in January, May, and September.
Yes, the 30-week PGCert in Responsible Artificial Intelligence is available for those seeking a shorter course.
The Programme Director for the MSc in Artificial Intelligence Online at the University of Hull is Dr. Rameez Kureshi.
The PGCert Artificial Intelligence program takes 30 weeks to complete on a part-time basis.
The total cost of the PGCert in Artificial Intelligence is £2,750.
The minimum academic qualification required to apply for the MSc Artificial Intelligence Online program is an Honours degree at 2:2 or above (or international equivalent) in a STEM discipline or a closely relate

## Define Questions

In [ ]:
questions = [
    "What is the mode of delivery for the MSc Artificial Intelligence at the University of Hull Online?",
    "How long does the MSc Artificial Intelligence course at the University of Hull Online take to complete?",
    "What is the total cost of the MSc Artificial Intelligence at the University of Hull Online?",
    "When are the start dates for the MSc Artificial Intelligence at the University of Hull Online?",
    "Is there a shorter AI course available besides the MSc Artificial Intelligence at the University of Hull Online?",
    "Who is the Programme Director for the MSc Artificial Intelligence at the University of Hull Online?",
    "How long does the PGCert in Artificial Intelligence take to complete at the University of Hull Online?",
    "What is the total cost of the PGCert in Artificial Intelligence at the University of Hull Online?",
    "What is the minimum academic qualification required to apply for the MSc Artificial Intelligence at the University of Hull Online?",
    "What if I don't have a degree but have relevant professional experience and want to apply to the University of Hull Online?",
    "What English language proficiency is required if my first language is not English for the University of Hull Online MSc Artificial Intelligence?",
    "Who are the lecturers for the MSc Artificial Intelligence at the University of Hull Online?",
    "What are the tuition payment options available at the University of Hull Online?",
    "How can I pay my tuition online at the University of Hull Online?",
    "How do I arrange to pay by bank transfer to the University of Hull Online?",
    "What is the WhatsApp number to reach the course advisers at the University of Hull Online?",
    "When should self-funding students make their tuition payments at the University of Hull Online?",
    "What is the tuition payment deadline for student loan users at the University of Hull Online?",
    "Where are students enrolled in the University of Hull Online programs from?",
    "What kind of student community can I expect as a learner at the University of Hull Online?",
    "What is my student status once I am accepted into a University of Hull Online master’s program?",
    "Will I receive a University of Hull student ID card as a University of Hull Online student?",
    "What will appear on my degree certificate after completing a master’s program at the University of Hull Online?",
    "Is there a graduation ceremony for students in the University of Hull Online programs?",
    "What is the phone number to contact course advisers for the University of Hull Online programs?",
    "What is the email address for enquiries about programs at the University of Hull Online?",
    "What are the tuition fee payment options available for the University of Hull Online programs?",
    "Is there a discount for referring a friend to the University of Hull Online?",
    "What is the total course fee for the MSc in Healthcare Leadership at the University of Hull Online?",
    "How much does the MA in Creative Writing cost at the University of Hull Online?",
    "What is the tuition fee for the PGDip in Healthcare Leadership at the University of Hull Online?",
    "What is the cost of the MSc in Dementia program at the University of Hull Online?",
    "How much does the PGCert in Healthcare Leadership cost at the University of Hull Online?",
    "What is the tuition fee for the PGDip in Dementia at the University of Hull Online?",
    "How much does the MSc in Logistics and Supply Chain Management cost at the University of Hull Online?",
    "What is the tuition fee for the PGCert in Dementia at the University of Hull Online?",
    "What is the total cost of the MSc in People Analytics at the University of Hull Online?",
    "How much does the MSc in Digital Transformation cost at the University of Hull Online?",
    "What is the tuition fee for the PG Award in People Analytics at the University of Hull Online?",
    "What is the total cost of the MA in Education at the University of Hull Online?",
    "How much does the MSc in Engineering Management cost at the University of Hull Online?",
    "How much does the Global MBA program cost at the University of Hull Online?",
    "What is the ranking of the University of Hull Online in the north of the UK according to the Times Good University Guide 2022?",
    "How did the University of Hull Online perform in the Guardian University Rankings 2022?",
    "What modules are covered in the MSc Artificial Intelligence at the University of Hull Online?",
    "What support is available if I have financial difficulties during the MSc Artificial Intelligence at the University of Hull Online?",
    "Are there exams in the MSc Artificial Intelligence at the University of Hull Online?",
    "What happens to my tuition fees if I temporarily suspend my studies at the University of Hull Online?",
    "What happens if I have outstanding financial obligations to the University of Hull Online?",
    "Why should I study the MSc Artificial Intelligence at the University of Hull Online?",
]
ground_truth_answers = [
    "The MSc Artificial Intelligence at the University of Hull Online is delivered 100% online.",
    "The MSc Artificial Intelligence at the University of Hull Online takes two years to complete on a part-time basis.",
    "The total cost of the MSc Artificial Intelligence at the University of Hull Online is £8,950, with instalment and funding options available.",
    "The start dates for the MSc Artificial Intelligence at the University of Hull Online are in January, May, and September.",
    "Yes, the PGCert in Responsible Artificial Intelligence is a shorter 30-week course offered at the University of Hull Online.",
    "The Programme Director for the MSc Artificial Intelligence at the University of Hull Online is Dr. Rameez Kureshi.",
    "The PGCert in Artificial Intelligence at the University of Hull Online takes 30 weeks to complete on a part-time basis.",
    "The total cost of the PGCert in Artificial Intelligence at the University of Hull Online is £2,750.",
    "To apply for the MSc Artificial Intelligence at the University of Hull Online, you need an Honours degree at 2:2 or above (or international equivalent) in a STEM field or related subject.",
    "If you don’t have a degree, you may still be considered for the MSc Artificial Intelligence at the University of Hull Online based on relevant professional experience and programming competence.",
    "To study the MSc Artificial Intelligence at the University of Hull Online, an IELTS score of 6.0 (with 5.5 minimum in each skill) or equivalent is required.",
    "The lecturers for the MSc Artificial Intelligence at the University of Hull Online include Dr. Rameez Kureshi, Professor Adil Khan, Khadija Fraz, Dr. Bhupesh Mishra, and Professor Dhaval Thakker.",
    "Tuition at the University of Hull Online can be paid in full, by instalments, via card, or bank transfer.",
    "To pay online for the University of Hull Online, you can use the Convera GlobalPay website after receiving your offer.",
    "To pay by bank transfer for the University of Hull Online, email finance-online@hull.ac.uk after receiving your offer.",
    "You can reach course advisers at the University of Hull Online via WhatsApp at +44 (0)7360 538906.",
    "Self-funding students at the University of Hull Online must pay four weeks before each term starts.",
    "Students using loans at the University of Hull Online must pay within two weeks of each loan disbursement.",
    "Students at the University of Hull Online come from over 100 countries around the world.",
    "As a student at the University of Hull Online, you will be part of a global, supportive learning community.",
    "Once accepted, you will become a fully registered student of the University of Hull Online.",
    "Yes, students enrolled in the University of Hull Online will receive a University of Hull student ID card.",
    "Your degree certificate from the University of Hull Online will not mention online study and will be identical to on-campus students.",
    "Yes, University of Hull Online students are invited to attend graduation ceremonies at the Hull campus.",
    "You can contact University of Hull Online course advisers by phone at +44(0)1482 251 819.",
    "The email address for general enquiries about the University of Hull Online is enquiries-online@hull.ac.uk.",
    "University of Hull Online students can pay tuition either in full or in six instalments over the course duration.",
    "Yes, students who refer a friend to the University of Hull Online may receive a discount of up to £750.",
    "The total course fee for the MSc in Healthcare Leadership at the University of Hull Online is £9,950.",
    "The MA in Creative Writing at the University of Hull Online costs £10,600.",
    "The tuition fee for the PGDip in Healthcare Leadership at the University of Hull Online is £6,700.",
    "The MSc in Dementia at the University of Hull Online costs £10,600.",
    "The PGCert in Healthcare Leadership at the University of Hull Online costs £3,400.",
    "The PGDip in Dementia at the University of Hull Online costs £7,100.",
    "The MSc in Logistics and Supply Chain Management at the University of Hull Online costs £9,950.",
    "The PGCert in Dementia at the University of Hull Online costs £3,600.",
    "The MSc in People Analytics at the University of Hull Online costs £10,600.",
    "The MSc in Digital Transformation at the University of Hull Online costs £10,600.",
    "The PG Award in People Analytics at the University of Hull Online costs £1,800.",
    "The MA in Education at the University of Hull Online costs £8,950.",
    "The MSc in Engineering Management at the University of Hull Online costs £8,950.",
    "The Global MBA at the University of Hull Online costs £12,150.",
    "The University of Hull Online is ranked 4th in the north of the UK according to the Times Good University Guide 2022.",
    "The University of Hull Online climbed 19 places in the Guardian University Rankings 2022.",
    "The MSc Artificial Intelligence at the University of Hull Online includes modules such as AI Foundations, Machine Learning & Deep Learning, Ethical AI, Applied AI, and a Research/Consultancy Project.",
    "If you have financial difficulties during the MSc Artificial Intelligence at the University of Hull Online, you may qualify for funding or bursaries. Contact +44(0)1482 251 819 or email enquiries-online@hull.ac.uk.",
    "There are no exams in the MSc Artificial Intelligence at the University of Hull Online; all assessments are coursework-based.",
    "If you suspend your studies during the MSc Artificial Intelligence at the University of Hull Online, fees already paid are retained and you may be liable for future charges if you resume.",
    "If you have unpaid fees at the University of Hull Online, you may not progress until the debt is resolved. Email finance-online@hull.ac.uk to dispute charges.",
    "The MSc Artificial Intelligence at the University of Hull Online offers flexibility, expert academic support, and part-time online study, preparing you for AI-driven careers.",
]


## A. Retrieval Extraction with 'all-'MiniLM'-v2' Embeddings and Cosine Similarity:

distilbert-base-cased-distilled-squad model is a question-answering model, not a sentence-transformers model, which is required for generating embeddings.

all-MiniLM-L6-v2' is a lightweight version of a transformer model designed for generating sentence embeddings, based on the architecture of transformer models like BERT, RoBERTa, DistilBERT.

"MiniLM" in the model name refers to Microsoft's MiniLM (Miniature Language Model).

**1. Embedding Process**:
all-MiniLM-L6-v2 used to generate sentence embeddings for both the questions and corresponding answers.
The embeddings created using SentenceTransformer, and cosine similarity is computed between question embedding and answer embeddings.

**2.Retrieval Process**:

For each question, computes cosine similarity between question's embedding and each answer's embedding.
The answer with highest cosine similarity score is selected as the most relevant answer.

**3. Evaluation**:

*Exact Match (EM) Score*calculated by comparing each retrieved answer to the ground truth answer. If they match exactly, it's considered a correct match.

*F1 Score*: This score measures the overlap between the tokens in the retrieved answer and the ground truth answer, considering both precision (how much of  retrieved answer is relevant) and recall (how much of relevant answer was retrieved).

**4.Visualization**:

EM and F1 scores are visualized

### A1. Embedding Process

In [ ]:
# Function to generate embeddings, find most relevant answers, save them

def perform_embeddings(embedding_model_name, questions, answers):
    model = SentenceTransformer(embedding_model_name) # Load embedding model
    answer_embeddings = model.encode(answers, convert_to_tensor=True)  # Generate embeddings for answers

    predictions = []

    for question in questions:
        question_embedding = model.encode(question, convert_to_tensor=True) # Generate embedding for the question
        similarities = util.pytorch_cos_sim(question_embedding, answer_embeddings)  # Compute cosine similarity between QnA
        most_similar_index = similarities.argmax() # Find the index of the most similar answer
        most_relevant_answer = answers[most_similar_index]  # Retrieve the most relevant answer

        predictions.append(most_relevant_answer)

        print(f"Q: {question}\nA: {most_relevant_answer}\n")

    return predictions, answer_embeddings

### A2.Retrieval Process:

In [ ]:
# Define model name
embedding_model_name = "all-MiniLM-L6-v2"

# Perform QA with MiniLM embeddings
predictions, MiniLM_embeddings = perform_embeddings(embedding_model_name, questions, ground_truth_answers)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q: What is the mode of delivery for the MSc Artificial Intelligence at the University of Hull Online?
A: The MSc Artificial Intelligence at the University of Hull Online is delivered 100% online.

Q: How long does the MSc Artificial Intelligence course at the University of Hull Online take to complete?
A: The MSc Artificial Intelligence at the University of Hull Online takes two years to complete on a part-time basis.

Q: What is the total cost of the MSc Artificial Intelligence at the University of Hull Online?
A: The total cost of the MSc Artificial Intelligence at the University of Hull Online is £8,950, with instalment and funding options available.

Q: When are the start dates for the MSc Artificial Intelligence at the University of Hull Online?
A: The start dates for the MSc Artificial Intelligence at the University of Hull Online are in January, May, and September.

Q: Is there a shorter AI course available besides the MSc Artificial Intelligence at the University of Hull Online

#### Save the "all-MiniLM-L6-v2" embeddings

In [ ]:
# Save embeddings
embedding_file_path = './data/MiniLM_embeddings_hull.pkl'

with open(embedding_file_path, 'wb') as f:
    pickle.dump(MiniLM_embeddings, f)

print(f"Embeddings saved to {embedding_file_path}")

Embeddings saved to ./data/MiniLM_embeddings_hull.pkl


#### Load and print the saved "all-MiniLM-L6-v2" embeddings from the file

In [ ]:
# Path to embeddings file
embedding_file_path = './data/MiniLM_embeddings_hull.pkl'

# Load embeddings from file
with open(embedding_file_path, 'rb') as f:
    loaded_embeddings = pickle.load(f)

print(f"Embeddings loaded from {embedding_file_path}:") # Print loaded embeddings
print(loaded_embeddings)

Embeddings loaded from ./data/MiniLM_embeddings_hull.pkl:
tensor([[-0.0112, -0.0451, -0.0221,  ...,  0.0533,  0.0426, -0.0064],
        [ 0.0040, -0.0335,  0.0653,  ...,  0.0207, -0.0161,  0.0055],
        [ 0.0273, -0.0479,  0.0087,  ..., -0.0894,  0.0107, -0.0260],
        ...,
        [-0.0463,  0.0032,  0.0127,  ..., -0.1100,  0.0223, -0.0848],
        [ 0.0120, -0.0218,  0.0228,  ..., -0.0977, -0.0191, -0.0406],
        [-0.0370, -0.0366,  0.0331,  ...,  0.0111,  0.0149,  0.0141]])


### A3. Evaluation:

### Calculates the Exact Match (EM) Score and F1 Score for a retrieval-based extraction task using all-MiniLM-L6-v2 embeddings and cosine similarity.

Exact Match (EM) Score: measures proportion of exactly matching answers between  retrievals and the ground truths.

F1 Score: Measures overlap between retrieved and ground truth answers in terms of shared tokens, balancing precision and recall.

In [ ]:
# Function to calculate Exact Match and F1 Score
def evaluate(predictions, ground_truths):
    em_score = sum([1 if pred == truth else 0 for pred, truth in zip(predictions, ground_truths)]) / len(ground_truths)

    f1_scores = []
    for pred, truth in zip(predictions, ground_truths):
        pred_tokens = pred.split()
        truth_tokens = truth.split()
        common_tokens = set(pred_tokens) & set(truth_tokens)
        if len(common_tokens) == 0:
            f1_scores.append(0)
        else:
            precision = len(common_tokens) / len(pred_tokens)
            recall = len(common_tokens) / len(truth_tokens)
            f1 = 2 * (precision * recall) / (precision + recall)
            f1_scores.append(f1)

    f1_score_avg = sum(f1_scores) / len(f1_scores)

    return em_score, f1_score_avg

# Evaluate the predictions
em_score, f1_score_avg = evaluate(predictions, ground_truth_answers)
print(f"Exact Match (EM) Score: {em_score:.2f}")
print(f"F1 Score: {f1_score_avg:.2f}")

Exact Match (EM) Score: 0.92
F1 Score: 0.90


### Interpretation:

Retrieval-based model with cosine similarity got high Exact Match (EM) Score and F1 Score for questions whose answers are available in the knowledge base

